In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# 1. Cargar el dataset usando el catálogo automático de Kedro
df = catalog.load("df_nhanes_ml_listo")

# 2. Separar características (X) y objetivo (y)
X = df.drop(columns=['Riesgo_Hipertension'])
y = df['Riesgo_Hipertension']

# 3. Dividir en Entrenamiento (80%) y Prueba (20%)
# Usamos stratify=y para que ambos sets tengan la misma proporción de 60/40 que descubrimos antes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Datos de entrenamiento: {X_train.shape[0]} pacientes")
print(f"Datos de prueba (ocultos): {X_test.shape[0]} pacientes")

[06/21/26 11:26:41] INFO     Loading data from df_nhanes_ml_listo (CSVDataset)...              ]8;id=14604774;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=14604775;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

Datos de entrenamiento: 16929 pacientes
Datos de prueba (ocultos): 4233 pacientes


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# 1. Inicializar los modelos
modelo_logistica = LogisticRegression(max_iter=1000, random_state=42)
modelo_forest = RandomForestClassifier(random_state=42, n_estimators=100)

# 2. Entrenar y predecir con Regresión Logística
print("⏳ Entrenando Regresión Logística...")
modelo_logistica.fit(X_train, y_train)
preds_logistica = modelo_logistica.predict(X_test)
probs_logistica = modelo_logistica.predict_proba(X_test)[:, 1]

# 3. Entrenar y predecir con Random Forest
print("⏳ Entrenando Random Forest (esto puede tardar unos segundos más)...")
modelo_forest.fit(X_train, y_train)
preds_forest = modelo_forest.predict(X_test)
probs_forest = modelo_forest.predict_proba(X_test)[:, 1]

# 4. Desplegar el reporte de métricas
print("\n" + "="*60)
print("🏆 RESULTADOS: REGRESIÓN LOGÍSTICA")
print("="*60)
print(classification_report(y_test, preds_logistica))
print(f"Área Bajo la Curva (ROC-AUC): {roc_auc_score(y_test, probs_logistica):.4f}")

print("\n" + "="*60)
print("🏆 RESULTADOS: RANDOM FOREST")
print("="*60)
print(classification_report(y_test, preds_forest))
print(f"Área Bajo la Curva (ROC-AUC): {roc_auc_score(y_test, probs_forest):.4f}")

⏳ Entrenando Regresión Logística...
⏳ Entrenando Random Forest (esto puede tardar unos segundos más)...

🏆 RESULTADOS: REGRESIÓN LOGÍSTICA
              precision    recall  f1-score   support

           0       0.72      0.77      0.74      2575
           1       0.60      0.53      0.56      1658

    accuracy                           0.68      4233
   macro avg       0.66      0.65      0.65      4233
weighted avg       0.67      0.68      0.67      4233

Área Bajo la Curva (ROC-AUC): 0.7387

🏆 RESULTADOS: RANDOM FOREST
              precision    recall  f1-score   support

           0       0.71      0.75      0.73      2575
           1       0.58      0.53      0.56      1658

    accuracy                           0.67      4233
   macro avg       0.65      0.64      0.64      4233
weighted avg       0.66      0.67      0.66      4233

Área Bajo la Curva (ROC-AUC): 0.7305


In [3]:
import numpy as np

# 1. Extraer los coeficientes del modelo de Regresión Logística
coeficientes = modelo_logistica.coef_[0]

# 2. Crear un DataFrame para mapear cada variable con su coeficiente
importancia_variables = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': coeficientes,
    'Impacto_Absoluto': np.abs(coeficientes) # Nos sirve para ordenar por fuerza total
})

# 3. Ordenar de mayor a menor impacto absoluto
importancia_variables = importancia_variables.sort_values(by='Impacto_Absoluto', ascending=False)
importancia_variables = importancia_variables.drop(columns=['Impacto_Absoluto']).reset_index(drop=True)

# 4. Mostrar el resultado completo
print("📊 IMPORTANCIA DE LAS VARIABLES EN LA PREDICCIÓN:")
print("="*60)
print(importancia_variables.to_string())

📊 IMPORTANCIA DE LAS VARIABLES EN LA PREDICCIÓN:
                    Variable  Coeficiente
0                    Peso_kg     1.163241
1                       Edad     0.831775
2                        IMC    -0.715567
3                Estatura_cm    -0.501133
4           Genero_Masculino     0.455761
5   Etnia_Mexicano Americano    -0.413542
6               Etnia_Blanco    -0.379412
7         Etnia_Otro Hispano    -0.323374
8           Colesterol_Total     0.253891
9                Etnia_Negro     0.206654
10   Etnia_Otra/Multirracial    -0.173674
11             Educacion_Cod    -0.067014
12            Ratio_Ingresos    -0.056201
13                Cintura_cm     0.043816
14         Categoria_IMC_Cod    -0.038154
15      Fumador_Historico_Sí    -0.030936


In [4]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Eliminamos las variables redundantes de los datos de entrenamiento y prueba
variables_redundantes = ['Peso_kg', 'Estatura_cm']
X_train_fix = X_train.drop(columns=variables_redundantes)
X_test_fix = X_test.drop(columns=variables_redundantes)

# 2. Re-entrenamos el modelo con los datos corregidos
modelo_final = LogisticRegression(max_iter=1000, random_state=42)
modelo_final.fit(X_train_fix, y_train)

# 3. En lugar de predecir 0 o 1 directamente, le pedimos la PROBABILIDAD exacta (de 0 a 1)
probabilidades_riesgo = modelo_final.predict_proba(X_test_fix)[:, 1]

# 4. Ajustamos el umbral médico: Si el paciente tiene 38% o más de probabilidad, va a riesgo (1)
umbral_medico = 0.38
predicciones_ajustadas = (probabilidades_riesgo >= umbral_medico).astype(int)

# ==========================================
# DESPLIEGUE DE RESULTADOS
# ==========================================

# Revisar cómo quedaron los coeficientes ahora
coef_df = pd.DataFrame({
    'Variable': X_train_fix.columns,
    'Nuevo_Coeficiente': modelo_final.coef_[0]
}).sort_values(by='Nuevo_Coeficiente', ascending=False).reset_index(drop=True)

print("🔄 REVISIÓN DE COEFICIENTES (¿Se arregló el IMC?):")
print("="*60)
print(coef_df.to_string())

print("\n" + "="*60)
print(f"🏆 METRICAS FINALES (Con Umbral Ajustado al {umbral_medico*100}%)")
print("="*60)
print(classification_report(y_test, predicciones_ajustadas))

🔄 REVISIÓN DE COEFICIENTES (¿Se arregló el IMC?):
                    Variable  Nuevo_Coeficiente
0                       Edad           0.818138
1           Genero_Masculino           0.475183
2                        IMC           0.276788
3           Colesterol_Total           0.256539
4                Etnia_Negro           0.193949
5                 Cintura_cm           0.106719
6       Fumador_Historico_Sí          -0.039405
7          Categoria_IMC_Cod          -0.041308
8             Ratio_Ingresos          -0.050747
9              Educacion_Cod          -0.066647
10   Etnia_Otra/Multirracial          -0.190001
11        Etnia_Otro Hispano          -0.338791
12              Etnia_Blanco          -0.391172
13  Etnia_Mexicano Americano          -0.440482

🏆 METRICAS FINALES (Con Umbral Ajustado al 38.0%)
              precision    recall  f1-score   support

           0       0.79      0.62      0.69      2575
           1       0.56      0.74      0.63      1658

    accuracy   

In [5]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Usamos los datos limpios de colinealidad que creamos en el paso anterior
# X_train_fix, X_test_fix, y_train, y_test

print("🔬 Iniciando optimización de Hiperparámetros... ¡Esto va a tardar un momento!")

# ==========================================
# 1. OPTIMIZACIÓN DE REGRESIÓN LOGÍSTICA
# ==========================================
print("\n⏳ Buscando los mejores hiperparámetros para Regresión Logística...")
param_grid_log = {
    'C': [0.01, 0.1, 1, 10, 100],  # Fuerza de la penalización
    'penalty': ['l2'],             # Tipo de regularización
    'solver': ['saga']             # Algoritmo de optimización eficiente
}

grid_log = GridSearchCV(
    LogisticRegression(max_iter=2000, random_state=42), 
    param_grid_log, cv=5, scoring='roc_auc', n_jobs=-1
)
grid_log.fit(X_train_fix, y_train)
best_log = grid_log.best_estimator_

# ==========================================
# 2. OPTIMIZACIÓN DE RANDOM FOREST
# ==========================================
print("⏳ Buscando los mejores hiperparámetros para Random Forest...")
param_grid_rf = {
    'n_estimators': [50, 100, 200],     # Número de árboles
    'max_depth': [5, 10, 20, None],      # Profundidad de los árboles
    'min_samples_split': [2, 5, 10]      # Muestras mínimas para dividir
}

grid_rf = GridSearchCV(
    RandomForestClassifier(random_state=42), 
    param_grid_rf, cv=5, scoring='roc_auc', n_jobs=-1
)
grid_rf.fit(X_train_fix, y_train)
best_rf = grid_rf.best_estimator_

# ==========================================
# 📊 EVALUACIÓN DE LOS CAMPEONES
# ==========================================
print("\n" + "="*60)
print("🏆 GANADORES Y SUS REGLAS OPTIMIZADAS")
print("="*60)
print(f"Mejores parámetros Logística: {grid_log.best_params_}")
print(f"ROC-AUC de Validación Logística: {grid_log.best_score_:.4f}\n")
print(f"Mejores parámetros Random Forest: {grid_rf.best_params_}")
print(f"ROC-AUC de Validación Random Forest: {grid_rf.best_score_:.4f}")

🔬 Iniciando optimización de Hiperparámetros... ¡Esto va a tardar un momento!

⏳ Buscando los mejores hiperparámetros para Regresión Logística...
⏳ Buscando los mejores hiperparámetros para Random Forest...

🏆 GANADORES Y SUS REGLAS OPTIMIZADAS
Mejores parámetros Logística: {'C': 0.1, 'penalty': 'l2', 'solver': 'saga'}
ROC-AUC de Validación Logística: 0.7453

Mejores parámetros Random Forest: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}
ROC-AUC de Validación Random Forest: 0.7468


In [6]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Usar el modelo campeón que ya está optimizado (best_log)
modelo_campeon = best_log

# 2. Calcular probabilidades exactas en el set de prueba (pacientes ocultos)
probabilidades_finales = modelo_campeon.predict_proba(X_test_fix)[:, 1]

# 3. Aplicar nuestro umbral preventivo (38%)
umbral_medico = 0.38
predicciones_finales = (probabilidades_finales >= umbral_medico).astype(int)

# ==========================================
# 📊 DESPLIEGUE DE RESULTADOS DEFINITIVOS
# ==========================================
print("\n" + "="*65)
print("🏆 REPORTE FINAL: REGRESIÓN LOGÍSTICA OPTIMIZADA (Umbral 38%)")
print("="*65)
print(classification_report(y_test, predicciones_finales))

# Calcular y desglosar la matriz de confusión
matriz = confusion_matrix(y_test, predicciones_finales)

print("\n" + "="*65)
print("🩺 MATRIZ DE CONFUSIÓN (Impacto en 4,233 pacientes reales)")
print("="*65)
print(f"✔️ Verdaderos Negativos (Sanos diagnosticados sanos): {matriz[0][0]}")
print(f"⚠️ Falsos Positivos (Sanos enviados a revisión por error): {matriz[0][1]}")
print(f"🚨 Falsos Negativos (Pacientes en riesgo ignorados): {matriz[1][0]}")
print(f"✔️ Verdaderos Positivos (Pacientes en riesgo detectados): {matriz[1][1]}")


🏆 REPORTE FINAL: REGRESIÓN LOGÍSTICA OPTIMIZADA (Umbral 38%)
              precision    recall  f1-score   support

           0       0.79      0.62      0.70      2575
           1       0.56      0.74      0.64      1658

    accuracy                           0.67      4233
   macro avg       0.67      0.68      0.67      4233
weighted avg       0.70      0.67      0.67      4233


🩺 MATRIZ DE CONFUSIÓN (Impacto en 4,233 pacientes reales)
✔️ Verdaderos Negativos (Sanos diagnosticados sanos): 1603
⚠️ Falsos Positivos (Sanos enviados a revisión por error): 972
🚨 Falsos Negativos (Pacientes en riesgo ignorados): 432
✔️ Verdaderos Positivos (Pacientes en riesgo detectados): 1226


In [7]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("="*80)
print("RESUMEN GENERAL DE MODELOS")
print("="*80)

# ==========================
# LOGISTIC REGRESSION
# ==========================
print("\n📌 LOGISTIC REGRESSION")

pred_log = modelo_logistica.predict(X_test)
prob_log = modelo_logistica.predict_proba(X_test)[:,1]

print(f"Accuracy : {accuracy_score(y_test, pred_log):.4f}")
print(f"Precision: {precision_score(y_test, pred_log):.4f}")
print(f"Recall   : {recall_score(y_test, pred_log):.4f}")
print(f"F1 Score : {f1_score(y_test, pred_log):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, prob_log):.4f}")

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, pred_log))

print("\nClassification Report:")
print(classification_report(y_test, pred_log))

# ==========================
# RANDOM FOREST
# ==========================
print("\n" + "="*80)
print("📌 RANDOM FOREST")

pred_rf = modelo_forest.predict(X_test)
prob_rf = modelo_forest.predict_proba(X_test)[:,1]

print(f"Accuracy : {accuracy_score(y_test, pred_rf):.4f}")
print(f"Precision: {precision_score(y_test, pred_rf):.4f}")
print(f"Recall   : {recall_score(y_test, pred_rf):.4f}")
print(f"F1 Score : {f1_score(y_test, pred_rf):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, prob_rf):.4f}")

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, pred_rf))

print("\nClassification Report:")
print(classification_report(y_test, pred_rf))

# ==========================
# LOGISTICA FINAL (38%)
# ==========================
print("\n" + "="*80)
print("📌 LOGISTICA FINAL UMBRAL 38%")

threshold = 0.38

prob_final = modelo_logistica.predict_proba(X_test)[:,1]
pred_final = (prob_final >= threshold).astype(int)

print(f"Accuracy : {accuracy_score(y_test, pred_final):.4f}")
print(f"Precision: {precision_score(y_test, pred_final):.4f}")
print(f"Recall   : {recall_score(y_test, pred_final):.4f}")
print(f"F1 Score : {f1_score(y_test, pred_final):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, prob_final):.4f}")

print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, pred_final))

print("\nClassification Report:")
print(classification_report(y_test, pred_final))

# ==========================
# GRID SEARCH
# ==========================
print("\n" + "="*80)
print("📌 GRID SEARCH RESULTS")

try:
    print("\nMejor Logistic Regression:")
    print("Best Params:", grid_log.best_params_)
    print("Best CV Score:", grid_log.best_score_)
except:
    print("grid_log no encontrado")

try:
    print("\nMejor Random Forest:")
    print("Best Params:", grid_rf.best_params_)
    print("Best CV Score:", grid_rf.best_score_)
except:
    print("grid_rf no encontrado")

print("\n" + "="*80)
print("FIN DEL REPORTE")
print("="*80)

RESUMEN GENERAL DE MODELOS

📌 LOGISTIC REGRESSION
Accuracy : 0.6756
Precision: 0.5967
Recall   : 0.5302
F1 Score : 0.5615
ROC AUC  : 0.7387

Matriz de Confusión:
[[1981  594]
 [ 779  879]]

Classification Report:
              precision    recall  f1-score   support

           0       0.72      0.77      0.74      2575
           1       0.60      0.53      0.56      1658

    accuracy                           0.68      4233
   macro avg       0.66      0.65      0.65      4233
weighted avg       0.67      0.68      0.67      4233


📌 RANDOM FOREST
Accuracy : 0.6664
Precision: 0.5810
Recall   : 0.5320
F1 Score : 0.5554
ROC AUC  : 0.7305

Matriz de Confusión:
[[1939  636]
 [ 776  882]]

Classification Report:
              precision    recall  f1-score   support

           0       0.71      0.75      0.73      2575
           1       0.58      0.53      0.56      1658

    accuracy                           0.67      4233
   macro avg       0.65      0.64      0.64      4233
weighted

In [8]:
print("Número de filas:", X_train.shape[0] + X_test.shape[0])
print("Número de variables:", X_train.shape[1])

for col in X_train.columns:
    print(col)

Número de filas: 21162
Número de variables: 16
Edad
Ratio_Ingresos
Peso_kg
Estatura_cm
IMC
Cintura_cm
Colesterol_Total
Categoria_IMC_Cod
Educacion_Cod
Genero_Masculino
Etnia_Blanco
Etnia_Mexicano Americano
Etnia_Negro
Etnia_Otra/Multirracial
Etnia_Otro Hispano
Fumador_Historico_Sí


In [12]:
import pandas as pd

df = pd.read_csv("data/03_primary/df_nhanes_ml_listo.csv")

print(df.columns.tolist())

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 import pandas as pd                                                                          │
│   2                                                                                              │
│ ❱ 3 df = pd.read_csv("data/03_primary/df_nhanes_ml_listo.csv")                                   │
│   4                                                                                              │
│   5 print(df.columns.tolist())                                                                   │
│   6                                                                                              │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\pandas\io\parsers\readers.py:10 │
│ 26 in read_csv                                                                                   │
│                                                                                                  │
│   1023 │   )                                                                                     │
│   1024 │   kwds.update(kwds_defaults)                                                            │
│   1025 │                                                                                         │
│ ❱ 1026 │   return _read(filepath_or_buffer, kwds)                                                │
│   1027                                                                                           │
│   1028                                                                                           │
│   1029 # iterator=True -> TextFileReader                                                         │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\pandas\io\parsers\readers.py:62 │
│ 0 in _read                                                                                       │
│                                                                                                  │
│    617 │   _validate_names(kwds.get("names", None))                                              │
│    618 │                                                                                         │
│    619 │   # Create the parser.                                                                  │
│ ❱  620 │   parser = TextFileReader(filepath_or_buffer, **kwds)                                   │
│    621 │                                                                                         │
│    622 │   if chunksize or iterator:                                                             │
│    623 │   │   return parser                                                                     │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\pandas\io\parsers\readers.py:16 │
│ 20 in __init__                                                                                   │
│                                                                                                  │
│   1617 │   │   │   self.options["has_index_names"] = kwds["has_index_names"]                     │
│   1618 │   │                                                                                     │
│   1619 │   │   self.handles: IOHandles | None = None                                             │
│ ❱ 1620 │   │   self._engine = self._make_engine(f, self.engine)                                  │
│   1621 │                                                   

In [13]:
%whos DataFrame

Variable                Type         Data/Info
----------------------------------------------
X                       DataFrame               Edad  Ratio_In<...>[21162 rows x 16 columns]
X_test                  DataFrame               Edad  Ratio_In<...>n[4233 rows x 16 columns]
X_test_fix              DataFrame               Edad  Ratio_In<...>n[4233 rows x 14 columns]
X_train                 DataFrame               Edad  Ratio_In<...>[16929 rows x 16 columns]
X_train_fix             DataFrame               Edad  Ratio_In<...>[16929 rows x 14 columns]
coef_df                 DataFrame                        Varia<...>ricano          -0.440482
df                      DataFrame               Edad  Ratio_In<...>[21162 rows x 17 columns]
importancia_variables   DataFrame                        Varia<...>Historico_Sí    -0.030936


In [14]:
# Ver todos los DataFrames cargados en memoria
for nombre, valor in globals().items():
    try:
        if hasattr(valor, 'shape'):
            print(f"{nombre}: {valor.shape}")
    except:
        pass

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 # Ver todos los DataFrames cargados en memoria                                               │
│ ❱ 2 for nombre, valor in globals().items():                                                      │
│   3 │   try:                                                                                     │
│   4 │   │   if hasattr(valor, 'shape'):                                                          │
│   5 │   │   │   print(f"{nombre}: {valor.shape}")                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
RuntimeError: dictionary changed size during iteration

In [10]:
print(y.value_counts())
print(y.value_counts(normalize=True))

Riesgo_Hipertension
0    12875
1     8287
Name: count, dtype: int64
Riesgo_Hipertension
0    0.608402
1    0.391598
Name: proportion, dtype: float64


In [15]:
import pandas as pd

df_merge = pd.read_parquet("data/02_intermediate/nhanes_merged.parquet")

print("Shape:", df_merge.shape)
print("\nPrimeras 50 columnas:")
print(df_merge.columns[:50].tolist())

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 import pandas as pd                                                                          │
│   2                                                                                              │
│ ❱ 3 df_merge = pd.read_parquet("data/02_intermediate/nhanes_merged.parquet")                     │
│   4                                                                                              │
│   5 print("Shape:", df_merge.shape)                                                              │
│   6 print("\nPrimeras 50 columnas:")                                                             │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\pandas\io\parquet.py:669 in     │
│ read_parquet                                                                                     │
│                                                                                                  │
│   666 │   │   use_nullable_dtypes = False                                                        │
│   667 │   check_dtype_backend(dtype_backend)                                                     │
│   668 │                                                                                          │
│ ❱ 669 │   return impl.read(                                                                      │
│   670 │   │   path,                                                                              │
│   671 │   │   columns=columns,                                                                   │
│   672 │   │   filters=filters,                                                                   │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\pandas\io\parquet.py:258 in     │
│ read                                                                                             │
│                                                                                                  │
│   255 │   │   manager = _get_option("mode.data_manager", silent=True)                            │
│   256 │   │   if manager == "array":                                                             │
│   257 │   │   │   to_pandas_kwargs["split_blocks"] = True                                        │
│ ❱ 258 │   │   path_or_handle, handles, filesystem = _get_path_or_handle(                         │
│   259 │   │   │   path,                                                                          │
│   260 │   │   │   filesystem,                                                                    │
│   261 │   │   │   storage_options=storage_options,                                               │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\pandas\io\parquet.py:141 in     │
│ _get_path_or_handle                                                                              │
│                                                                                                  │
│   138 │   │   # use get_handle only when we are very certain that it is not a directory          │
│   139 │   │   # fsspec resources can also point to directories                                   │
│   140 │   │   # this branch is used for example when reading from non-fsspec URLs                │
│ ❱ 141 │   │   handles = get_handle(                                                              │
│   142 │   │   │   path_or_handle, mode, is_text=False, stor

In [16]:
# 1. Cargar el dataset inicial fusionado desde el catálogo de Kedro
df_inicial = catalog.load("nhanes_merged")

# 2. Ver cuántas filas (pacientes) y columnas (variables) totales tiene originalmente
print(f"Dimensiones originales: {df_inicial.shape[0]} filas y {df_inicial.shape[1]} columnas.")

# 3. Mostrar una lista con los nombres de TODAS las variables originales
print("\n📋 LISTA DE TODAS LAS VARIABLES ORIGINALES:")
print("="*60)
for col in df_inicial.columns:
    print(col)

# 4. Ver las primeras 5 filas para espiar cómo venían los datos crudos
df_inicial.head()

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│    1 # 1. Cargar el dataset inicial fusionado desde el catálogo de Kedro                         │
│ ❱  2 df_inicial = catalog.load("nhanes_merged")                                                  │
│    3                                                                                             │
│    4 # 2. Ver cuántas filas (pacientes) y columnas (variables) totales tiene originalmente       │
│    5 print(f"Dimensiones originales: {df_inicial.shape[0]} filas y {df_inicial.shape[1]} colu    │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py:1051   │
│ in load                                                                                          │
│                                                                                                  │
│   1048 │   │                                                                                     │
│   1049 │   │   if dataset is None:                                                               │
│   1050 │   │   │   error_msg = f"Dataset '{ds_name}' not found in the catalog"                   │
│ ❱ 1051 │   │   │   raise DatasetNotFoundError(error_msg)                                         │
│   1052 │   │                                                                                     │
│   1053 │   │   self._logger.info(                                                                │
│   1054 │   │   │   "Loading data from %s (%s)...",                                               │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
DatasetNotFoundError: Dataset 'nhanes_merged' not found in the catalog

In [17]:
# Buscar en el catálogo todos los datasets que tengan "merged" o "nhanes" en su nombre
nombres_disponibles = [ds for ds in catalog.list() if "merged" in ds.lower() or "nhanes" in ds.lower()]

print("Nombres registrados en tu catálogo de Kedro:")
print("="*60)
for nombre in nombres_disponibles:
    print(f"- {nombre}")

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:2                                                                                    │
│                                                                                                  │
│   1 # Buscar en el catálogo todos los datasets que tengan "merged" o "nhanes" en su nombre       │
│ ❱ 2 nombres_disponibles = [ds for ds in catalog.list() if "merged" in ds.lower() or "nhanes"     │
│   3                                                                                              │
│   4 print("Nombres registrados en tu catálogo de Kedro:")                                        │
│   5 print("="*60)                                                                                │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
AttributeError: 'DataCatalogWithCatalogCommandsMixin' object has no attribute 'list'

In [18]:
# 1. Buscamos usando list(catalog) en lugar de catalog.list()
nombres_disponibles = [ds for ds in list(catalog) if "merged" in ds.lower() or "nhanes" in ds.lower()]

print("Nombres registrados en tu catálogo de Kedro:")
print("="*60)
for nombre in nombres_disponibles:
    print(f"- {nombre}")

Nombres registrados en tu catálogo de Kedro:
- nhanes_merged_data
- df_nhanes_limpio
- df_nhanes_ml_listo


In [19]:
# 2. Sustituye 'EL_NOMBRE_QUE_VISTE_ARRIBA' por el resultado que te dio el código anterior
df_inicial = catalog.load("nhanes_merged_data")

# Ver cuántas filas (pacientes) y columnas (variables) totales tiene originalmente
print(f"Dimensiones originales: {df_inicial.shape[0]} filas y {df_inicial.shape[1]} columnas.")

# Mostrar las primeras filas y explorar el mar de variables crudas
df_inicial.head()

[06/21/26 12:14:44] INFO     Loading data from nhanes_merged_data (ParquetDataset)...          ]8;id=14604780;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=14604781;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

Dimensiones originales: 15456 filas y 149 columnas.


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGY,...,SMQ925,SMQ930,SMQ935,SMQ080,SMQ890,SMQ895,SMQ900,SMQ905,SMQ910,SMQ915
0,83732,9.0,2.0,1.0,62.0,NaN,3.0,3.0,1.0,NaN,...,NaN,NaN,NaN,NaN,2.0,NaN,2.0,NaN,1.0,5.397605e-79
1,83733,9.0,2.0,1.0,53.0,NaN,3.0,3.0,1.0,NaN,...,NaN,NaN,NaN,NaN,1.0,3.000000e+01,2.0,NaN,2.0,NaN
2,83734,9.0,2.0,1.0,78.0,NaN,3.0,3.0,2.0,NaN,...,NaN,NaN,NaN,NaN,1.0,5.397605e-79,2.0,NaN,1.0,5.397605e-79
3,83735,9.0,2.0,2.0,56.0,NaN,3.0,3.0,2.0,NaN,...,1.0,7.0,3.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN
4,83736,9.0,2.0,2.0,42.0,NaN,4.0,4.0,2.0,NaN,...,2.0,NaN,NaN,NaN,1.0,7.000000e+00,2.0,NaN,2.0,NaN


In [20]:
print("Dimensiones:", nhanes_merged_data.shape)

palabras_clave = [
    "BP",      # Presión arterial
    "DIQ",     # Diabetes
    "GLU",     # Glucosa
    "GH",      # Hemoglobina glicosilada
    "CHOL",    # Colesterol
    "TRIG",    # Triglicéridos
    "HDL",     # HDL
    "PAQ",     # Actividad física
    "ALQ",     # Alcohol
    "SLD",     # Sueño
    "SMQ",     # Tabaquismo
    "MCQ",     # Enfermedades
    "BMX"      # Medidas corporales
]

for palabra in palabras_clave:
    cols = [c for c in nhanes_merged_data.columns if palabra.upper() in c.upper()]

    if cols:
        print(f"\n===== {palabra} =====")
        for c in cols:
            print(c)

print("\nNúmero total de columnas:", len(nhanes_merged_data.columns))

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱  1 print("Dimensiones:", nhanes_merged_data.shape)                                             │
│    2                                                                                             │
│    3 palabras_clave = [                                                                          │
│    4 │   "BP",      # Presión arterial                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
NameError: name 'nhanes_merged_data' is not defined

In [21]:
print("Dimensiones:", df_inicial.shape)

# Buscar variables potencialmente útiles para hipertensión
palabras_clave = [
    "BP",      # presión arterial
    "DIQ",     # diabetes
    "GLU",     # glucosa
    "GH",      # hemoglobina glicosilada
    "CHOL",    # colesterol
    "TRIG",    # triglicéridos
    "HDL",     # colesterol HDL
    "PAQ",     # actividad física
    "ALQ",     # alcohol
    "SLD",     # sueño
    "SMQ",     # tabaquismo
    "MCQ",     # enfermedades
    "BMX",     # medidas corporales
]

for palabra in palabras_clave:
    cols = [c for c in df_inicial.columns if palabra.upper() in c.upper()]
    if cols:
        print(f"\n===== {palabra} =====")
        print(cols[:30])  # mostrar hasta 30

Dimensiones: (15456, 149)

===== BP =====
['BPXCHR', 'BPAARM', 'BPACSZ', 'BPXPLS', 'BPXPULS', 'BPXPTY', 'BPXML1', 'BPXSY1', 'BPXDI1', 'BPAEN1', 'BPXSY2', 'BPXDI2', 'BPAEN2', 'BPXSY3', 'BPXDI3', 'BPAEN3', 'BPXSY4', 'BPXDI4', 'BPAEN4']

===== SMQ =====
['SMQ020', 'SMQ040', 'SMQ050Q', 'SMQ050U', 'SMQ077', 'SMQ621', 'SMQ660', 'SMQ664M', 'SMQ664C', 'SMQ664W', 'SMQ664B', 'SMQ664O', 'SMQ670', 'SMQ078', 'SMQ661', 'SMQ665A', 'SMQ665B', 'SMQ665C', 'SMQ665D', 'SMQ848', 'SMQ852Q', 'SMQ852U', 'SMQ925', 'SMQ930', 'SMQ935', 'SMQ080', 'SMQ890', 'SMQ895', 'SMQ900', 'SMQ905']

===== BMX =====
['BMXWT', 'BMXRECUM', 'BMXHEAD', 'BMXHT', 'BMXBMI', 'BMXLEG', 'BMXARML', 'BMXARMC', 'BMXWAIST', 'BMXSAD1', 'BMXSAD2', 'BMXSAD3', 'BMXSAD4', 'BMXHIP']


In [22]:
# 1. Cargamos el dataset usando su "apodo" en el catálogo
df_inicial = catalog.load("nhanes_merged_data")

# 2. Ver cuántas filas (pacientes) y columnas (variables) totales tiene originalmente
print(f"Dimensiones originales: {df_inicial.shape[0]} filas y {df_inicial.shape[1]} columnas.")

# 3. Mostrar las primeras 5 filas para explorar el mar de variables crudas
df_inicial.head()

[06/21/26 12:20:07] INFO     Loading data from nhanes_merged_data (ParquetDataset)...          ]8;id=14604786;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=14604787;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

Dimensiones originales: 15456 filas y 149 columnas.


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGY,...,SMQ925,SMQ930,SMQ935,SMQ080,SMQ890,SMQ895,SMQ900,SMQ905,SMQ910,SMQ915
0,83732,9.0,2.0,1.0,62.0,NaN,3.0,3.0,1.0,NaN,...,NaN,NaN,NaN,NaN,2.0,NaN,2.0,NaN,1.0,5.397605e-79
1,83733,9.0,2.0,1.0,53.0,NaN,3.0,3.0,1.0,NaN,...,NaN,NaN,NaN,NaN,1.0,3.000000e+01,2.0,NaN,2.0,NaN
2,83734,9.0,2.0,1.0,78.0,NaN,3.0,3.0,2.0,NaN,...,NaN,NaN,NaN,NaN,1.0,5.397605e-79,2.0,NaN,1.0,5.397605e-79
3,83735,9.0,2.0,2.0,56.0,NaN,3.0,3.0,2.0,NaN,...,1.0,7.0,3.0,NaN,2.0,NaN,2.0,NaN,2.0,NaN
4,83736,9.0,2.0,2.0,42.0,NaN,4.0,4.0,2.0,NaN,...,2.0,NaN,NaN,NaN,1.0,7.000000e+00,2.0,NaN,2.0,NaN


In [23]:
# 1. Ver qué ciclos únicos están guardados en la columna de ciclos de NHANES
if 'SDDSRVYR' in df_inicial.columns:
    ciclos_presentes = df_inicial['SDDSRVYR'].unique()
    print("============================================================")
    print("🔍 VERIFICACIÓN DE CICLOS EN EL DATASET INICIAL")
    print("============================================================")
    print(f"Los códigos de los ciclos encontrados son: {ciclos_presentes}")
    print(f"En total tienes: {len(ciclos_presentes)} ciclos fusionados aquí.")
else:
    print("No se encontró la columna SDDSRVYR, vamos a listar las primeras columnas para buscarla.")
    print(df_inicial.columns[:10].tolist())

🔍 VERIFICACIÓN DE CICLOS EN EL DATASET INICIAL
Los códigos de los ciclos encontrados son: [ 9. 10.]
En total tienes: 2 ciclos fusionados aquí.


In [1]:
# Sacar la lista completa de todo lo que existe en el catálogo
todos_los_nombres = list(catalog)

print("📋 TODOS LOS DATASETS REGISTRADOS EN KEDRO:")
print("="*60)
for nombre in todos_los_nombres:
    # Filtramos los parámetros internos de Kedro para que sea más fácil leer
    if not nombre.startswith("params:") and not nombre.startswith("parameters"):
        print(f"👉 {nombre}")

📋 TODOS LOS DATASETS REGISTRADOS EN KEDRO:
👉 demographics_raw
👉 body_measures_sql
👉 laboratory_api
👉 nhanes_merged_data
👉 df_nhanes_limpio
👉 df_nhanes_ml_listo


In [2]:
# 1. Cargar la fuente original de demografía y el archivo final de ML
df_demo = catalog.load("demographics_raw")
df_ml = catalog.load("df_nhanes_ml_listo")

# 2. Imprimir la realidad de los tamaños
print("📊 VERIFICACIÓN DE TAMAÑOS EN EL PIPELINE:")
print("="*60)
print(f"1. Origen Crudo (demographics_raw): {df_demo.shape[0]} pacientes.")
print(f"2. Archivo Final (df_nhanes_ml_listo): {df_ml.shape[0]} pacientes.")

[06/21/26 17:03:29] INFO     Loading data from demographics_raw (CSVDataset)...                ]8;id=15287554;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=15287555;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

                    INFO     Loading data from df_nhanes_ml_listo (CSVDataset)...              ]8;id=15287560;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=15287561;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

📊 VERIFICACIÓN DE TAMAÑOS EN EL PIPELINE:
1. Origen Crudo (demographics_raw): 39156 pacientes.
2. Archivo Final (df_nhanes_ml_listo): 21162 pacientes.


In [3]:
# 1. Cargar las otras dos fuentes de datos originales
df_body = catalog.load("body_measures_sql")
df_lab = catalog.load("laboratory_api")

# 2. Imprimir los tamaños para comparar
print("🔍 AUDITORÍA DE FUENTES ORIGINALES:")
print("="*60)
print(f"1. Demografía (demographics_raw): {catalog.load('demographics_raw').shape[0]} filas.")
print(f"2. Medidas Corporales (body_measures_sql): {df_body.shape[0]} filas.")
print(f"3. Laboratorio (laboratory_api): {df_lab.shape[0]} filas.")

[06/21/26 17:14:16] INFO     Loading data from body_measures_sql (SQLTableDataset)...          ]8;id=15287566;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=15287567;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\sqlalchemy\engine\base.py:144   │
│ in __init__                                                                                      │
│                                                                                                  │
│    141 │   │                                                                                     │
│    142 │   │   if connection is None:                                                            │
│    143 │   │   │   try:                                                                          │
│ ❱  144 │   │   │   │   self._dbapi_connection = engine.raw_connection()                          │
│    145 │   │   │   except dialect.loaded_dbapi.Error as err:                                     │
│    146 │   │   │   │   Connection._handle_dbapi_exception_noconnection(                          │
│    147 │   │   │   │   │   err, dialect, engine                                                  │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\sqlalchemy\engine\base.py:3319  │
│ in raw_connection                                                                                │
│                                                                                                  │
│   3316 │   │   │   :ref:`dbapi_connections`                                                      │
│   3317 │   │                                                                                     │
│   3318 │   │   """                                                                               │
│ ❱ 3319 │   │   return self.pool.connect()                                                        │
│   3320                                                                                           │
│   3321                                                                                           │
│   3322 class OptionEngineMixin(log.Identified):                                                  │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\sqlalchemy\pool\base.py:448 in  │
│ connect                                                                                          │
│                                                                                                  │
│    445 │   │   the pool.                                                                         │
│    446 │   │                                                                                     │
│    447 │   │   """                                                                               │
│ ❱  448 │   │   return _ConnectionFairy._checkout(self)                                           │
│    449 │                                                                                         │
│    450 │   def _return_conn(self, record: ConnectionPoolEntry) -> None:                          │
│    451 │   │   """Given a _ConnectionRecord, return it to the :class:`_pool.Pool`.               │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\sqlalchemy\pool\base.py:1272 in │
│ _checkout                                                                                        │
│                                                                                                  │
│   1269 │   │   fairy: Optional[_ConnectionFairy] = None,                                         │
│   1270 │   ) -> _ConnectionFairy:                                                                │
│   1271 │   │   if not fairy:                               

In [4]:
df_body = catalog.load("body_measures_sql")
print(f"Medidas Corporales: {df_body.shape[0]} filas.")

[06/21/26 17:18:57] INFO     Loading data from body_measures_sql (SQLTableDataset)...          ]8;id=15287572;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py\data_catalog.py]8;;\:]8;id=15287573;file://C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\kedro\io\data_catalog.py#1053\1053]8;;\

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\sqlalchemy\engine\base.py:144   │
│ in __init__                                                                                      │
│                                                                                                  │
│    141 │   │                                                                                     │
│    142 │   │   if connection is None:                                                            │
│    143 │   │   │   try:                                                                          │
│ ❱  144 │   │   │   │   self._dbapi_connection = engine.raw_connection()                          │
│    145 │   │   │   except dialect.loaded_dbapi.Error as err:                                     │
│    146 │   │   │   │   Connection._handle_dbapi_exception_noconnection(                          │
│    147 │   │   │   │   │   err, dialect, engine                                                  │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\sqlalchemy\engine\base.py:3319  │
│ in raw_connection                                                                                │
│                                                                                                  │
│   3316 │   │   │   :ref:`dbapi_connections`                                                      │
│   3317 │   │                                                                                     │
│   3318 │   │   """                                                                               │
│ ❱ 3319 │   │   return self.pool.connect()                                                        │
│   3320                                                                                           │
│   3321                                                                                           │
│   3322 class OptionEngineMixin(log.Identified):                                                  │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\sqlalchemy\pool\base.py:448 in  │
│ connect                                                                                          │
│                                                                                                  │
│    445 │   │   the pool.                                                                         │
│    446 │   │                                                                                     │
│    447 │   │   """                                                                               │
│ ❱  448 │   │   return _ConnectionFairy._checkout(self)                                           │
│    449 │                                                                                         │
│    450 │   def _return_conn(self, record: ConnectionPoolEntry) -> None:                          │
│    451 │   │   """Given a _ConnectionRecord, return it to the :class:`_pool.Pool`.               │
│                                                                                                  │
│ C:\Users\maico\Proyectos\nhanes-pipeline\.venv\lib\site-packages\sqlalchemy\pool\base.py:1272 in │
│ _checkout                                                                                        │
│                                                                                                  │
│   1269 │   │   fairy: Optional[_ConnectionFairy] = None,                                         │
│   1270 │   ) -> _ConnectionFairy:                                                                │
│   1271 │   │   if not fairy:                               